In [512]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [513]:
data = pd.read_csv('../data/processed/matches.csv')
data.head()

,date,home_team,away_team,tournament,city,country,neutral,year,month,day,...,stadium_temperature_max,stadium_temperature_min,stadium_precipitation,stadium_wind_speed,home_stadium_temp_max,home_stadium_temp_min,home_stadium_temp_avg,away_stadium_temp_max,away_stadium_temp_min,away_stadium_temp_avg
0,1994-06-18,Italy,Republic of Ireland,FWC,East Rutherford,United States,True,1994,6,18,...,32.2,21.3,0.4,11.2,6.7,6.8,6.75,12.8,9.3,11.05
1,1994-06-19,Belgium,Morocco,FWC,Orlando,United States,True,1994,6,19,...,30.1,21.9,6.1,11.0,7.2,10.6,8.90,4.4,6.0,5.20
2,1994-06-21,Argentina,Greece,FWC,Foxborough,United States,True,1994,6,21,...,24.9,16.8,1.0,17.9,10.9,8.4,9.65,10.9,4.9,7.90
3,1994-06-21,Germany,Spain,FWC,Chicago,United States,True,1994,6,21,...,26.7,22.3,1.7,21.9,5.0,12.8,8.90,3.7,5.7,4.70
4,1994-06-22,Romania,Switzerland,FWC,Pontiac,United States,True,1994,6,22,...,27.8,15.2,0.0,10.2,3.0,1.5,2.25,5.0,3.6,4.30


In [514]:
data.drop(columns=['tournament', 'city', 'country', 
                   'year', 'date', 'home_capital', 'away_capital', 
                   'home_comb', 'away_comb', 'stadium_comb', 'home_team', 'away_team',
                   "home_lon", "home_lat", "away_lon", "away_lat",
                   "stadium_lon", "stadium_lat"], inplace=True)

In [515]:
data.columns

Index(['neutral', 'month', 'day', 'result', 'home_points', 'away_points',
       'home_stadium_distance_km', 'away_stadium_distance_km', 'home_fix_1',
       'home_fix_2', 'home_shots_1', 'home_shots_2', 'home_shots_against_1',
       'home_shots_against_2', 'home_scored_1', 'home_scored_2',
       'home_conceded_1', 'home_conceded_2', 'home_relative_shots_1',
       'home_relative_shots_2', 'home_relative_goals_1',
       'home_relative_goals_2', 'away_fix_1', 'away_fix_2', 'away_shots_1',
       'away_shots_2', 'away_shots_against_1', 'away_shots_against_2',
       'away_scored_1', 'away_scored_2', 'away_conceded_1', 'away_conceded_2',
       'away_relative_shots_1', 'away_relative_shots_2',
       'away_relative_goals_1', 'away_relative_goals_2', 'home_ranking',
       'away_ranking', 'home_temperature_max', 'home_temperature_min',
       'home_precipitation', 'home_wind_speed', 'away_temperature_max',
       'away_temperature_min', 'away_precipitation', 'away_wind_speed',
       's

In [516]:
data.select_dtypes(include=['object']).columns

Index([], dtype='str')

In [517]:
data["home_stadium_wind_speed"] = np.abs(data["home_wind_speed"] - data["stadium_wind_speed"])
data["away_stadium_wind_speed"] = np.abs(data["away_wind_speed"] - data["stadium_wind_speed"])
data["ranking_diff"] = data["home_ranking"] - data["away_ranking"]
data["stadium_temperature_avg"] = (data["stadium_temperature_max"] + data["stadium_temperature_min"]) / 2
data["home_temperature_avg"] = (data["home_temperature_max"] + data["home_temperature_min"]) / 2
data["away_temperature_avg"] = (data["away_temperature_max"] + data["away_temperature_min"]) / 2


data["home_relative_goals_1"] = data["home_scored_1"] - data["home_conceded_1"]
data["home_relative_goals_2"] = data["home_scored_2"] - data["home_conceded_2"]

data["away_relative_goals_1"] = data["away_scored_1"] - data["away_conceded_1"]
data["away_relative_goals_2"] = data["away_scored_2"] - data["away_conceded_2"]

In [518]:
data.columns

Index(['neutral', 'month', 'day', 'result', 'home_points', 'away_points',
       'home_stadium_distance_km', 'away_stadium_distance_km', 'home_fix_1',
       'home_fix_2', 'home_shots_1', 'home_shots_2', 'home_shots_against_1',
       'home_shots_against_2', 'home_scored_1', 'home_scored_2',
       'home_conceded_1', 'home_conceded_2', 'home_relative_shots_1',
       'home_relative_shots_2', 'home_relative_goals_1',
       'home_relative_goals_2', 'away_fix_1', 'away_fix_2', 'away_shots_1',
       'away_shots_2', 'away_shots_against_1', 'away_shots_against_2',
       'away_scored_1', 'away_scored_2', 'away_conceded_1', 'away_conceded_2',
       'away_relative_shots_1', 'away_relative_shots_2',
       'away_relative_goals_1', 'away_relative_goals_2', 'home_ranking',
       'away_ranking', 'home_temperature_max', 'home_temperature_min',
       'home_precipitation', 'home_wind_speed', 'away_temperature_max',
       'away_temperature_min', 'away_precipitation', 'away_wind_speed',
       's

In [519]:
data.drop(columns=["home_points", "away_points",
                   "stadium_temperature_max", "stadium_temperature_min",
                   "home_temperature_max", "home_temperature_min",
                   "away_temperature_max", "away_temperature_min",
                   "home_stadium_temp_max", "home_stadium_temp_min",
                   "away_stadium_temp_max", "away_stadium_temp_min"], inplace=True)

In [520]:
from sklearn.model_selection import train_test_split

In [521]:
train, val = train_test_split(data, test_size=0.2, random_state=42)

# Feature Scaling

In [522]:
from sklearn.preprocessing import RobustScaler

In [ ]:
X_train = train.drop(columns=['result']).copy()
y_train = train['result']

X_val = val.drop(columns=['result']).copy()
y_val = val['result']

# Cyclic encoding
for df in [X_train, X_val]:
    df["day_sin"] = np.sin(2 * np.pi * df["day"] / 31)
    df["day_cos"] = np.cos(2 * np.pi * df["day"] / 31)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Drop the original day/month columns
X_train = X_train.drop(columns=["day", "month"])
X_val = X_val.drop(columns=["day", "month"])


cyclic_cols = ["day_sin", "day_cos", "month_sin", "month_cos"]
scale_cols = [c for c in X_train.columns if c not in cyclic_cols]

scaler = RobustScaler()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])

X_train_columns = X_train.columns


In [544]:
import joblib
joblib.dump(scaler, '../models/robust_scaler.pkl')

['../models/robust_scaler.pkl']

In [525]:
scaled_train = pd.concat([
    X_train.reset_index(drop=True),
    y_train.reset_index(drop=True)
], axis=1)

scaled_val = pd.concat([
    X_val.reset_index(drop=True),
    y_val.reset_index(drop=True)
], axis=1)

# Feature Selection

## Correlation

In [526]:
columns = scaled_train.columns
for i in range(len(columns)):
    for col in columns[i + 1:]:
        if scaled_train[columns[i]].corr(scaled_train[col]) > 0.8:
            print("High correlation between {} and {} with {}".format(columns[i], col, scaled_train[columns[i]].corr(scaled_train[col])))

High correlation between home_fix_2 and home_relative_goals_2 with 0.8093962807052357
High correlation between away_fix_1 and away_relative_goals_1 with 0.8038702110742124
High correlation between away_fix_2 and away_relative_goals_2 with 0.8093067594151387


In [527]:
# scaled_train.drop(columns=["home_points", "away_points", "home_shots_2",
#                    "home_shots_1", "away_shots_1", "away_shots_2",
#                    "home_temperature_max", "home_temperature_min",
#                    "away_temperature_max", "away_temperature_min",
#                    "stadium_temperature_max", "stadium_temperature_min",
#                    "home_stadium_temp_max", "home_stadium_temp_min",
#                    "away_stadium_temp_max", "away_stadium_temp_min"], inplace=True)


In [528]:
# scaled_train.drop(columns=["home_points", "away_points",
#                    "stadium_temperature_max", "stadium_temperature_min",
#                    "home_temperature_max", "home_temperature_min",
#                    "away_temperature_max", "away_temperature_min",
#                    "home_stadium_temp_max", "home_stadium_temp_min",
#                    "away_stadium_temp_max", "away_stadium_temp_min"], inplace=True)

In [529]:
scaled_train.columns

Index(['neutral', 'home_stadium_distance_km', 'away_stadium_distance_km',
       'home_fix_1', 'home_fix_2', 'home_shots_1', 'home_shots_2',
       'home_shots_against_1', 'home_shots_against_2', 'home_scored_1',
       'home_scored_2', 'home_conceded_1', 'home_conceded_2',
       'home_relative_shots_1', 'home_relative_shots_2',
       'home_relative_goals_1', 'home_relative_goals_2', 'away_fix_1',
       'away_fix_2', 'away_shots_1', 'away_shots_2', 'away_shots_against_1',
       'away_shots_against_2', 'away_scored_1', 'away_scored_2',
       'away_conceded_1', 'away_conceded_2', 'away_relative_shots_1',
       'away_relative_shots_2', 'away_relative_goals_1',
       'away_relative_goals_2', 'home_ranking', 'away_ranking',
       'home_precipitation', 'home_wind_speed', 'away_precipitation',
       'away_wind_speed', 'stadium_precipitation', 'stadium_wind_speed',
       'home_stadium_temp_avg', 'away_stadium_temp_avg',
       'home_stadium_wind_speed', 'away_stadium_wind_speed', 'ra

# Feature Selection

## Variance Threshold

In [530]:
from sklearn.feature_selection import RFECV

## RFE

In [531]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

In [532]:
X = scaled_train.drop(columns=["result"])
y = scaled_train["result"]

In [533]:
estimator_cv = RandomForestClassifier(random_state=42)
rfecv = RFECV(
    estimator=estimator_cv,
    step=1,
    cv=StratifiedKFold(3),      # 3 folds × n_features iterations
    scoring="accuracy",
)
rfecv.fit(X, y)

selected_features_rfecv = X.columns[rfecv.support_].tolist()

print(f"Total CV iterations: {X.shape[1] * 3}")   # n_features × n_folds
print(f"Optimal n_features:  {rfecv.n_features_}")
print(f"\nSelected features ({len(selected_features_rfecv)}):")
for f in selected_features_rfecv:
    print(f"  ✓ {f}")

print(f"\nCV accuracy per n_features (mean):")
cv_scores = pd.Series(rfecv.cv_results_['mean_test_score'],
                      index=range(1, X.shape[1] + 1))
print(cv_scores.to_string())
print(f"\nBest score: {cv_scores.max():.4f} at n={cv_scores.idxmax()} features")

Total CV iterations: 153
Optimal n_features:  23

Selected features (23):
  ✓ away_stadium_distance_km
  ✓ home_fix_1
  ✓ home_fix_2
  ✓ home_shots_against_1
  ✓ home_shots_against_2
  ✓ home_relative_shots_1
  ✓ home_relative_shots_2
  ✓ away_fix_1
  ✓ away_fix_2
  ✓ away_relative_shots_1
  ✓ away_relative_shots_2
  ✓ home_ranking
  ✓ away_ranking
  ✓ home_wind_speed
  ✓ away_wind_speed
  ✓ stadium_wind_speed
  ✓ home_stadium_temp_avg
  ✓ away_stadium_temp_avg
  ✓ away_stadium_wind_speed
  ✓ ranking_diff
  ✓ stadium_temperature_avg
  ✓ home_temperature_avg
  ✓ away_temperature_avg

CV accuracy per n_features (mean):
1     0.325814
2     0.430605
3     0.461732
4     0.468830
5     0.447566
6     0.451881
7     0.466036
8     0.466030
9     0.497193
10    0.473104
11    0.477329
12    0.480190
13    0.473116
14    0.475947
15    0.474528
16    0.481596
17    0.473086
18    0.470249
19    0.484445
20    0.475959
21    0.484433
22    0.481626
23    0.508529
24    0.497229
25    0.467472


In [534]:
scaled_train_selected = scaled_train[selected_features_rfecv + ["result"]]
scaled_val_selected = scaled_val[selected_features_rfecv + ["result"]]

scaled_train_selected.to_csv('../data/processed/scaled_train.csv', index=False)
scaled_val_selected.to_csv('../data/processed/scaled_val.csv', index=False)

In [536]:
selected_features_rfecv

['away_stadium_distance_km',
 'home_fix_1',
 'home_fix_2',
 'home_shots_against_1',
 'home_shots_against_2',
 'home_relative_shots_1',
 'home_relative_shots_2',
 'away_fix_1',
 'away_fix_2',
 'away_relative_shots_1',
 'away_relative_shots_2',
 'home_ranking',
 'away_ranking',
 'home_wind_speed',
 'away_wind_speed',
 'stadium_wind_speed',
 'home_stadium_temp_avg',
 'away_stadium_temp_avg',
 'away_stadium_wind_speed',
 'ranking_diff',
 'stadium_temperature_avg',
 'home_temperature_avg',
 'away_temperature_avg']